# Classifying Target Detection
<span style="font-family: 'Helvetica'; font-size: 15px;">
    Target detection is two-stage. We partition the dataset into mutual pairs and independent targets. We define a mutual pair as follows: two targets within a drop whose closest neighbor is each other (we ignore the scenario when more than two are identically close). All other targets are classified as independent. The reason for doing so is because the case of two, very close targets has interesting statistical properties that shouldn't be captured with a single model used for all other targets. Namely, It is very likely that of the two, close targets, only one of them is detected, but the probability that both are detected, or both are missed, is very low. This conditional probability cannot be recreated by sampling from their marginal distribution.
</span>

In [13]:
import sys
import polars as pl
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.components.detection import IndepClassifier, FullClassifier, PairClassifier

### Independent Classifier

<span style="font-family: 'Helvetica'; font-size: 15px;">
    The independent classifier simply uses a LightGBM for binary classification. All features are under IndepFeature in src/data/features.py. To train, simply instantialize an IndepClassifier object and then call .fit() on your training data. Note that it's best practice to split train and validation set using group_split(), which ensures entire drops are wthin either the train, or validation set to avoid cross contamination. However, doing a normal train test split (tt_split()) doesn't seem to produce much difference.
</span>

In [14]:
from src.data.load import load_indep, group_split

tags, labels, features = load_indep()
[X, X_val, y, y_val] = group_split(tags, labels, features)

classifier = IndepClassifier()
classifier.fit(X, y)


/home/owen-0g/sensing-abstraction/.venv/lib/python3.12/site-packages/lightgbm/basic.py:359: UserWarning: Converting column-vector to 1d array
  _log_warning("Converting column-vector to 1d array")


<span style="font-family: 'Helvetica'; font-size: 15px;">
    We can also choose to include callbacks, or functions that run during training via the lightgbm api. For example, we can pass callbacks to print the accuracy during each training step, and also choose to end training after a certain number of rounds pass without improvement to loss. Finally, we can pass in custom parameters for our model. 
</span>

In [15]:
import lightgbm as lgb
INDEP_PARAMS = {
    "objective": "binary",
    "metric": ["auc", "average_precision"],
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.65,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,

    #Use cuda to speed up (can ignore if on CPU)
    'device': 'cuda',       
    'gpu_use_dp': False,
}

CALLBACKS = [
    # stops after 200 rounds of no improvement
    lgb.early_stopping(stopping_rounds = 200), 
    lgb.log_evaluation(period = 20)
]

classifier_custom = IndepClassifier()
classifier_custom.fit(X, y, X_val, y_val, params = INDEP_PARAMS, callbacks=CALLBACKS)


Training until validation scores don't improve for 200 rounds
[20]	train's auc: 0.987614	train's average_precision: 0.998988	valid's auc: 0.97881	valid's average_precision: 0.998205
[40]	train's auc: 0.990315	train's average_precision: 0.999216	valid's auc: 0.979396	valid's average_precision: 0.998259
[60]	train's auc: 0.992467	train's average_precision: 0.999395	valid's auc: 0.979987	valid's average_precision: 0.998314
[80]	train's auc: 0.994347	train's average_precision: 0.999548	valid's auc: 0.980886	valid's average_precision: 0.998401
[100]	train's auc: 0.995919	train's average_precision: 0.999675	valid's auc: 0.980872	valid's average_precision: 0.998395
Did not meet early stopping. Best iteration is:
[99]	train's auc: 0.995852	train's average_precision: 0.99967	valid's auc: 0.980986	valid's average_precision: 0.998406


<span style="font-family: 'Helvetica'; font-size: 15px;">
    We can save and load the model using the appropriate methods to avoid having to retrain. At inference time, we can use the predict_proba() method to get the probabilities of detection, and then sample a detected/not detected using sample()
</span>

In [16]:
import numpy as np

classifier_custom.save(path = "models/indep_classifier.joblib")
classifier_custom = IndepClassifier.load(path = "models/indep_classifier.joblib")
preds = classifier_custom.predict_proba(X_val)
rng = np.random.default_rng(seed = 42)

samples = classifier_custom.sample(preds, rng)


### Pair Classifier

<span style="font-family: 'Helvetica'; font-size: 15px;">
The pair classifier functions differently in that it is trained on the joint probability of mutual pairs. After identifying mutual pairs, process_pair_features joins the two rows containing each target into a single row. The first target is labeled _a, and the second _b, and the columns contain each corresponding feature suffixed with _a, or _b, except for shared features such as nn_norm_range_sep. Then, the data is duplicated to force symmetry about the a, b labeling by switching the a, b index. The label is a score 0, 1, 2, 3 based on the following rule: the pair receives 0 if neither are detected, 1 if only b is detected, 2 if only a, and 3 if both are detected. Pre processing the data for pair processor should be taken care of with process_pair_features and should not be done manually.
</span>

In [17]:
from src.data.load import load_pairs

PAIR_PARAMS = {
    "objective": "multiclass",
    "num_class" : 4,
    "metric": "multi_logloss",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.65,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "verbosity": -1,
    "seed": 42,

    #Use cuda to speed up (can ignore if on CPU)
    'device': 'cuda',       
    'gpu_use_dp': False,
}

tags, labels, features = load_pairs()
[X, X_val, y, y_val] = group_split(tags, labels, features)

pair_classifier = PairClassifier()
pair_classifier.fit(X, y, X_val, y_val, params = PAIR_PARAMS, callbacks=CALLBACKS)
pair_classifier.save(path = "models/pair_classifier.joblib")

/home/owen-0g/sensing-abstraction/.venv/lib/python3.12/site-packages/lightgbm/basic.py:359: UserWarning: Converting column-vector to 1d array
  _log_warning("Converting column-vector to 1d array")


Training until validation scores don't improve for 200 rounds
[20]	train's multi_logloss: 0.422126	valid's multi_logloss: 0.45781
[40]	train's multi_logloss: 0.321282	valid's multi_logloss: 0.376509
[60]	train's multi_logloss: 0.268435	valid's multi_logloss: 0.34145
[80]	train's multi_logloss: 0.234477	valid's multi_logloss: 0.323926
[100]	train's multi_logloss: 0.208779	valid's multi_logloss: 0.313947
Did not meet early stopping. Best iteration is:
[100]	train's multi_logloss: 0.208779	valid's multi_logloss: 0.313947


<span style="font-family: 'Helvetica'; font-size: 15px;">
    The training sequence followed largely the same sequence as the independent classifier. Except now, the output will be an array of probabilities corresponding to each class. Sampling will take a random class based off of these probabilities, and output a dataframe with rows corresponding to each pair; whether a was detected and whether b was detected
</span>

In [18]:
pair_classifier = PairClassifier.load(path = "models/pair_classifier.joblib")
preds = pair_classifier.predict_proba(X_val)
samples = pair_classifier.sample(preds, rng)
print(samples)

shape: (5_814, 2)
┌────────────┬────────────┐
│ a_detected ┆ b_detected │
│ ---        ┆ ---        │
│ bool       ┆ bool       │
╞════════════╪════════════╡
│ true       ┆ false      │
│ false      ┆ true       │
│ false      ┆ true       │
│ true       ┆ false      │
│ true       ┆ true       │
│ …          ┆ …          │
│ true       ┆ true       │
│ true       ┆ true       │
│ true       ┆ false      │
│ true       ┆ true       │
│ true       ┆ true       │
└────────────┴────────────┘


### Full Classifier

<span style="font-family: 'Helvetica'; font-size: 15px;">
    The full classifier abstracts away any manual processing, splitting, and recombining the indep, mutual pair training split. Any full, production training/inference classification pipeline should only use the Full Classifier class. The inputs come from the process_classifier_features() function that builds from the full feature dataset assembled. Note that every data point needs both the features from IndepFeature and PairFeature because the model cannot know which target is which until after it splits. 
</span>



In [19]:
from src.data.load import load_class

tags, labels, features = load_class()
[X, X_val, y, y_val, t, t_val] = group_split(tags, labels, features, return_tags=True)


full_classifier = FullClassifier()
train_df = pl.concat([X,y,t], how = "horizontal")
val_df = pl.concat([X_val, y_val, t_val], how = "horizontal")
full_classifier.fit(train_df, val_df)
full_classifier.save("models/full_classifier.joblib")

/tmp/ipykernel_1854/4087648737.py:8: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  train_df = pl.concat([X,y,t], how = "horizontal")
/tmp/ipykernel_1854/4087648737.py:9: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  val_df = pl.concat([X_val, y_val, t_val], how = "horizontal")
/home/owen-0g/sensing-abstraction/.venv/lib/python3.12/site-packages/lightgbm/basic.py:359: UserWarning: Converting column-vector to 1d array
  _log_warning("Converting column-vector to 1d array")


Training until validation scores don't improve for 200 rounds
[20]	train's auc: 0.987584	train's average_precision: 0.998983	valid's auc: 0.982927	valid's average_precision: 0.998685
[40]	train's auc: 0.990376	train's average_precision: 0.99922	valid's auc: 0.983065	valid's average_precision: 0.99869
[60]	train's auc: 0.992656	train's average_precision: 0.999409	valid's auc: 0.983326	valid's average_precision: 0.998712
[80]	train's auc: 0.994403	train's average_precision: 0.999551	valid's auc: 0.98405	valid's average_precision: 0.998776
[100]	train's auc: 0.995909	train's average_precision: 0.999673	valid's auc: 0.984331	valid's average_precision: 0.998799
Did not meet early stopping. Best iteration is:
[100]	train's auc: 0.995909	train's average_precision: 0.999673	valid's auc: 0.984331	valid's average_precision: 0.998799
Training until validation scores don't improve for 200 rounds
[20]	train's multi_logloss: 0.420735	valid's multi_logloss: 0.462433
[40]	train's multi_logloss: 0.3198


<span style="font-family: 'Helvetica'; font-size: 15px;">
    The full classifier performs a split of the independent targets and mutual pairs before dispatching traiing to its member variables, self.indep_model and self.pair_model. We need to pass the tags here in order for the classifier to determine how to split. At inference time, the classifier performs an indep_mutual_split() on the data given, and then dispatches inference to the two models it trained, returning the probability of detection for the independent predictions, and the joint probability for the mutual pairs
</span>



In [20]:
full_classifier = FullClassifier.load("models/full_classifier.joblib")
inf_df = pl.concat([X_val, t_val], how = "horizontal")
indep_preds, pair_preds = full_classifier.predict_proba(inf_df)


/tmp/ipykernel_1854/1763246607.py:2: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  inf_df = pl.concat([X_val, t_val], how = "horizontal")



<span style="font-family: 'Helvetica'; font-size: 15px;">
    To sample, the classifier once again dispatches each probability output into its corresponding sampler. To reassemble the outputs into a format that corresponds to the features given, use the reassemble() method. The first argument should be the original inference dataset used, its tags are needed for realigning the indep, mutual outputs. This returns back the original dataset given for training with the sampled detection under the "detected" column.
</span>



In [21]:
indep_samples, pair_samples = full_classifier.sample(indep_preds, pair_preds, rng)
output = full_classifier.reassemble(inf_df, indep_samples, pair_samples)
print(output["detected"])

shape: (9_914,)
Series: 'detected' [bool]
[
	true
	true
	false
	true
	true
	…
	true
	true
	true
	true
	true
]
